# Day 3-1: MNIST 데이터 탐색 & 베이스라인 모델

**강의 시간**: 1.5시간  
**학습 목표**:
- Kaggle 플랫폼 이해 및 대회 참여 방법 학습
- MNIST 손글씨 데이터셋 구조 파악
- 탐색적 데이터 분석 (EDA) 수행
- 데이터 전처리 파이프라인 구축
- Simple CNN 베이스라인 모델 구현
- MLflow로 첫 실험 기록

**사전 요구사항**: Day 2 완료, Kaggle 계정  
**예상 성능**: Validation Accuracy ~98%

## 🏆 0. Kaggle 플랫폼 소개

### 0.1 Kaggle이란?

**세계 최대의 데이터 사이언스 경진대회 플랫폼**

- 2010년 설립, 2017년 Google 인수
- 1000만+ 데이터 과학자 커뮤니티
- 다양한 대회: 상금, 채용, 학습 목적

### 0.2 Digit Recognizer 대회

**초보자를 위한 입문 대회**
- 문제: 손글씨 숫자 인식 (0-9)
- 평가: Accuracy (정확도)
- 특징: 제출 무제한, 리더보드 공개

**참여 방법:**
1. https://www.kaggle.com/c/digit-recognizer 접속
2. "Join Competition" 클릭
3. Rules 동의
4. Data 탭에서 train.csv, test.csv 다운로드

## 🔧 1. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout
)
from tensorflow.keras.utils import to_categorical

# MLflow
import mlflow
import dagshub

# 시각화 설정
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# 재현성
np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# Dagshub & MLflow 설정
repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(
    repo_owner=repo_owner,
    repo_name=repo_name,
    mlflow=True
)

mlflow.set_experiment('day3-mnist-digit-recognizer')
print('✅ Dagshub 연동 완료!')


## 📂 2. 데이터 로드

### 2.1 데이터 다운로드

Kaggle에서 직접 다운로드:

- https://www.kaggle.com/c/digit-recognizer/data
    - train.csv, test.csv 다운로드
- Colab에 업로드
    - 혹은 Google Drive에 업로드

#### Google Drive 연동

실험 결과와 데이터를 저장하기 위해 Google Drive를 연결합니다.

🔐 **실행하면 인증 링크가 나타납니다. 클릭해서 권한을 승인하세요.**

In [ ]:
# from google.colab import drive

# # Google Drive 마운트
# drive.mount('/content/drive')

# print("\n✅ Google Drive 연결 완료!")
# print("📁 Drive 경로: /content/drive/MyDrive")

In [ ]:
# import os

# # 폴더 구조 생성
# # base_path = '/content/drive/MyDrive/deeplearning-bootcamp'
# base_path = '/content/drive/MyDrive/lectures/dl_bootcamp'
# day3_path = os.path.join(base_path, 'day3_mnist_digit_recognizer')
# data_path = os.path.join(day3_path, 'data/digit-recognizer')

# os.makedirs(data_path, exist_ok=True)

# print("✅ 폴더 생성 완료!")
# print(f"📁 Base: {base_path}")
# print(f"📁 Day 3: {day3_path}")
# print(f"📁 Data: {data_path}")

#### 직접 업로드

In [ ]:
import os
import zipfile
from google.colab import files

# 1. 경로 설정: /content/data/digit-recognizer 폴더 생성
# 다른 데이터와 섞이지 않게 전용 하위 폴더를 지정합니다.
base_data_path = '/content/data'
target_path = os.path.join(base_data_path, 'digit-recognizer')

os.makedirs(target_path, exist_ok=True)

# 2. 파일 업로드
print("📤 'digit-recognizer.zip' 파일을 선택해주세요...")
uploaded = files.upload()

# 3. 압축 해제 로직
zip_file_name = 'digit-recognizer.zip'

if zip_file_name in uploaded:
    print(f"\n📦 {zip_file_name}을(를) {target_path}에 압축 해제 중...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        # target_path(/content/data/digit-recognizer)에 압축 해제
        zip_ref.extractall(target_path)
    print(f"✅ 압축 해제 완료: {target_path}")

    # 세션 용량 확보를 위해 업로드된 zip 파일 삭제 (선택 사항)
    os.remove(zip_file_name)
else:
    print(f"\n⚠️ {zip_file_name} 파일이 업로드되지 않았습니다.")

# 4. 결과 확인
print(f"\n📂 {target_path} 내부 파일 목록:")
print(os.listdir(target_path))

In [ ]:
data_path = target_path

#### 데이터 로드

In [ ]:
# 데이터 로드
# Kaggle에서 직접 다운로드하여 Google Drive에 업로드한 파일을 로드합니다.
train_df = pd.read_csv(os.path.join(data_path, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'test.csv'))

print(f"✅ Train shape: {train_df.shape}")
print(f"✅ Test shape : {test_df.shape}")
print()
print("Train 첫 5행:")
train_df.head()

In [ ]:
# 데이터 구조 확인
print("=" * 60)
print("  데이터 구조")
print("=" * 60)
print(f"Train: {train_df.shape[0]:,}개 샘플 × {train_df.shape[1]:,}개 컬럼")
print(f"  - label: 1개 (정답)")
print(f"  - pixels: 784개 (28×28 이미지)")
print()
print(f"Test: {test_df.shape[0]:,}개 샘플 × {test_df.shape[1]:,}개 컬럼")
print(f"  - pixels: 784개만 (label 없음)")
print("=" * 60)

# Label 분리
y_train = train_df['label'].values
X_train = train_df.drop('label', axis=1).values
X_test = test_df.values

print()
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}")

## 🔍 3. 탐색적 데이터 분석 (EDA)

In [ ]:
# 클래스 분포 확인
class_counts = pd.Series(y_train).value_counts().sort_index()

plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.index, class_counts.values,
               color='steelblue', edgecolor='black')
plt.title('Class Distribution (Train Set)', fontweight='bold', fontsize=14)
plt.xlabel('Digit', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(range(10))
plt.grid(axis='y', alpha=0.3)

# 값 표시
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 50,
            f'{int(height)}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("클래스별 개수:")
print(class_counts)
print()
print("💡 관찰: 비교적 균형 잡힌 분포 (클래스 불균형 문제 없음)")

In [ ]:
# 샘플 이미지 시각화 (각 클래스별 5개씩)
fig, axes = plt.subplots(10, 5, figsize=(10, 18))

for digit in range(10):
    # 해당 클래스 샘플 5개
    digit_samples = X_train[y_train == digit][:5]

    for i in range(5):
        ax = axes[digit, i]
        img = digit_samples[i].reshape(28, 28)
        ax.imshow(img, cmap='gray')
        ax.axis('off')

        if i == 0:
            ax.set_ylabel(f'Digit {digit}', fontsize=12, fontweight='bold')

plt.suptitle('Sample Images (5 per class)', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# 픽셀 값 분포 분석
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 전체 픽셀 히스토그램
axes[0].hist(X_train.flatten(), bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Pixel Value Distribution (All)', fontweight='bold')
axes[0].set_xlabel('Pixel Value (0-255)')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

# 비영 픽셀만 (값 > 0)
nonzero_pixels = X_train[X_train > 0]
axes[1].hist(nonzero_pixels, bins=50, color='coral', edgecolor='black')
axes[1].set_title('Non-zero Pixel Distribution', fontweight='bold')
axes[1].set_xlabel('Pixel Value (1-255)')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Zero pixels: {(X_train == 0).sum():,} ({(X_train == 0).mean()*100:.1f}%)")
print(f"Non-zero pixels: {(X_train > 0).sum():,} ({(X_train > 0).mean()*100:.1f}%)")
print()
print("💡 관찰: 대부분 0 (배경) + 일부 높은 값 (숫자 획)")

In [ ]:
# 혼동하기 쉬운 숫자 쌍 비교
confusing_pairs = [(7, 1), (4, 9), (3, 8), (5, 6)]

fig, axes = plt.subplots(len(confusing_pairs), 10, figsize=(15, len(confusing_pairs)*1.5))

for row_idx, (digit1, digit2) in enumerate(confusing_pairs):
    samples1 = X_train[y_train == digit1][:5]
    samples2 = X_train[y_train == digit2][:5]

    for i in range(5):
        # digit1
        ax = axes[row_idx, i]
        ax.imshow(samples1[i].reshape(28, 28), cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_ylabel(f'{digit1} vs {digit2}', fontsize=11, fontweight='bold')

        # digit2
        ax = axes[row_idx, i+5]
        ax.imshow(samples2[i].reshape(28, 28), cmap='gray')
        ax.axis('off')

plt.suptitle('Confusing Digit Pairs', fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

print("💡 이 숫자들은 모델이 혼동하기 쉬움:")
print("  - 7 vs 1: 획이 짧으면 구분 어려움")
print("  - 4 vs 9: 윗부분 닫혀 있으면 혼동")
print("  - 3 vs 8: 중간 부분 겹치면 혼동")
print("  - 5 vs 6: 회전되어 있으면 혼동")

## ⚙️ 4. 데이터 전처리

🔥 이 부분을 같이 작성해봅시다.

픽셀 값을 **0~255**에서 **0.0~1.0**으로 정규화해 보세요.

In [ ]:
# 정규화 (0-255 → 0.0-1.0)
X_train = # 🔥 직접 작성이 필요합니다.
X_test  = # 🔥 직접 작성이 필요합니다.

print(f'✅ 정규화 완료')
print(f'   Min: {X_train.min():.4f}')
print(f'   Max: {X_train.max():.4f}')


🔥 이 부분을 같이 작성해봅시다.

CNN 입력 형식 `(N, H, W, C)` = `(N, 28, 28, 1)`로 **Reshape**해 보세요.

In [ ]:
# Reshape (CNN 입력 형식)
X_train = # 🔥 직접 작성이 필요합니다. (reshape(-1, 28, 28, 1))
X_test  = # 🔥 직접 작성이 필요합니다.

print(f'✅ Reshape 완료')
print(f'   X_train: {X_train.shape} (N, H, W, C)')
print(f'   X_test : {X_test.shape}')


🔥 이 부분을 같이 작성해봅시다.

**train_test_split**으로 Train/Validation을 분리해 보세요. (test_size=0.1, stratify 적용)

In [ ]:
# Train/Validation Split
X_train_sub, X_val, y_train_sub, y_val = # 🔥 직접 작성이 필요합니다.

print(f'✅ Train/Val Split 완료')
print(f'   Train: {X_train_sub.shape[0]:,}개')
print(f'   Val  : {X_val.shape[0]:,}개')


## 🏗️ 5. Simple CNN 베이스라인

### 5.1 아키텍처 설계

```mermaid
graph TD
    A["Input<br/>(28, 28, 1)"] --> B["Conv2D(32, 3×3)<br/>+ ReLU"]
    B --> C["MaxPooling2D<br/>(2×2)"]
    C --> D["Conv2D(64, 3×3)<br/>+ ReLU"]
    D --> E["MaxPooling2D<br/>(2×2)"]
    E --> F["Flatten"]
    F --> G["Dense(128)<br/>+ ReLU"]
    G --> H["Dropout(0.5)"]
    H --> I["Dense(10)<br/>+ Softmax"]
    
    style A fill:#e1f5fe
    style B fill:#fff9c4
    style C fill:#fff9c4
    style D fill:#fff9c4
    style E fill:#fff9c4
    style F fill:#f3e5f5
    style G fill:#c8e6c9
    style H fill:#c8e6c9
    style I fill:#ffccbc
```


**특징:**
- 2개 Conv layer (feature extraction)
- 2개 Pooling (downsampling)
- 1개 Dense layer (classification)
- Dropout (regularization)

🔥 이 부분을 같이 작성해봅시다.

**Sequential** 안에 Conv Block 1 (Conv2D(32) + MaxPooling), Conv Block 2 (Conv2D(64) + MaxPooling), Flatten, Dense(128), Dropout(0.5), Dense(10, softmax) 레이어를 채워보세요.

In [ ]:
def build_simple_cnn():
    """Simple CNN 베이스라인 모델"""
    model = Sequential([
        # Conv Block 1
        # 🔥 직접 작성이 필요합니다. (Conv2D(32, ...) + MaxPooling2D)

        # Conv Block 2
        # 🔥 직접 작성이 필요합니다. (Conv2D(64, ...) + MaxPooling2D)

        # Classifier
        # 🔥 직접 작성이 필요합니다. (Flatten + Dense(128) + Dropout(0.5) + Dense(10, softmax))
    ])
    return model

model = build_simple_cnn()
model.summary()


In [ ]:
# 모델 컴파일
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 모델 컴파일 완료!")

## 🚀 6. 모델 학습

🔥 이 부분은 수정이 필요합니다.

**run_name**을 실험을 구분하기 쉬운 이름으로 채운 뒤 실행하고, Dagshub UI에서 학습 곡선을 확인해 보세요.

In [ ]:
# MLflow 실험 시작
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다.

    # 하이퍼파라미터 로깅
    params = {
        'model': 'Simple_CNN',
        'conv1_filters': 32,
        'conv2_filters': 64,
        'dense_units': 128,
        'dropout': 0.5,
        'optimizer': 'adam',
        'batch_size': 128,
        'epochs': 10,
    }
    mlflow.log_params(params)

    # 학습
    print("🏃 학습 시작...")
    history = model.fit(
        X_train_sub, y_train_sub,
        batch_size=128,
        epochs=10,
        validation_data=(X_val, y_val),
        verbose=1
    )

    # 메트릭 로깅
    for epoch in range(10):
        mlflow.log_metrics({
            'train_loss': history.history['loss'][epoch],
            'train_accuracy': history.history['accuracy'][epoch],
            'val_loss': history.history['val_loss'][epoch],
            'val_accuracy': history.history['val_accuracy'][epoch],
        }, step=epoch)

    # 최종 성능
    final_val_acc = history.history['val_accuracy'][-1]
    mlflow.log_metric('final_val_accuracy', final_val_acc)

    print(f"\n✅ 학습 완료!")
    print(f"   Final Validation Accuracy: {final_val_acc:.4f}")

    run_id = mlflow.active_run().info.run_id
    print(f"   MLflow Run ID: {run_id}")

## 📊 7. 결과 분석

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs = range(1, 11)

# Loss
axes[0].plot(epochs, history.history['loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs, history.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_title('Loss Curves', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, history.history['accuracy'], 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs, history.history['val_accuracy'], 'r-', label='Val Acc', linewidth=2)
axes[1].set_title('Accuracy Curves', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Simple CNN Baseline — 학습 결과', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

# MLflow 아티팩트 로깅
mlflow.log_artifact('training_curves.png')

In [ ]:
# 최종 평가
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print("=" * 60)
print("  최종 성능 (Validation Set)")
print("=" * 60)
print(f"  Loss    : {val_loss:.4f}")
print(f"  Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
print("=" * 60)
print()
print("💡 목표 달성: ~98% Accuracy ✅" if val_acc >= 0.98 else "⚠️ 목표 미달: 추가 학습 필요")

In [ ]:
model

In [ ]:
import mlflow.keras
from mlflow.models.signature import infer_signature
import numpy as np
import os

# 1. 모델 로컬 저장
model_save_path = 'simple_cnn_baseline.keras'
model.save(model_save_path)

# 2. 서명 및 예시 데이터 준비
# 입력을 딕셔너리 형태로 감싸면 MLflow가 더 안정적으로 처리합니다.
input_example = X_train_sub[0:1].astype(np.float32)
prediction = model.predict(input_example)
signature = infer_signature(input_example, prediction)

try:
    # 3. MLflow 모델 로깅
    # 만약 계속 에러가 난다면 input_example을 잠시 제거하고 signature만 넣어보세요.
    mlflow.keras.log_model(
        model=model,
        artifact_path="model",
        signature=signature,
        # input_example=input_example # <--- 여전히 에러가 나면 이 줄을 주석 처리하세요.
    )
    print("✅ MLflow: model artifact 등록 완료 (with signature)")

except Exception as e:
    print(f"❌ MLflow 로깅 중 에러 발생: {e}")
    print("💡 팁: 'input_example' 옵션을 빼고 다시 시도해보세요.")

print(f"✅ 로컬 파일 저장 완료: {model_save_path}")

## 🧠 8. 핵심 개념 정리

### 오늘 배운 것

**Kaggle 플랫폼**
- 세계 최대 ML 경진대회 플랫폼
- Digit Recognizer: 초보자용 입문 대회
- 리더보드, 무제한 제출, 커뮤니티 학습

**MNIST 데이터**
- 손글씨 숫자 (0-9), 28×28 grayscale
- Train 42,000개, Test 28,000개
- 균형 잡힌 클래스 분포

**EDA 중요성**
- 클래스 분포 → 불균형 확인
- 샘플 시각화 → 데이터 품질 확인
- 픽셀 분포 → 정규화 필요성 파악
- 혼동 케이스 → 모델 난이도 예측

**데이터 전처리**
```python
정규화: X / 255.0 (0-1 범위)
Reshape: (-1, 28, 28, 1) CNN 입력 형식
Split: 90% Train / 10% Val (stratify)
```

**Simple CNN**
- 2 Conv + 2 Pool + 1 Dense
- ~225K 파라미터
- 98% Validation Accuracy
- 좋은 출발점!

**MLflow 실험 관리**
- 하이퍼파라미터 자동 기록
- 메트릭 시계열 로깅
- 모델 & 아티팩트 저장
- 실험 비교 가능

---

### Day 3-2 예고

**다양한 CNN 아키텍처 비교**
1. LeNet-5 (1998, 역사적)
2. VGG-style (깊은 네트워크)
3. ResNet-style (Skip Connection)
4. Attention-based (SE, CBAM)
5. Custom Hybrid (Conv + Self-Attention)

**목표**: 98% → 99%+ 성능 향상

## ✅ Day 3-1 완료 체크리스트

- [ ] Kaggle 계정 생성 및 Digit Recognizer 대회 참여
- [ ] train.csv, test.csv 다운로드
- [ ] 데이터 로드 및 구조 확인
- [ ] EDA: 클래스 분포, 샘플 이미지, 픽셀 분포
- [ ] 혼동 케이스 분석 (7 vs 1, 4 vs 9 등)
- [ ] 데이터 전처리 (정규화, Reshape, Split)
- [ ] Simple CNN 모델 구현
- [ ] 모델 학습 (10 epochs)
- [ ] 학습 곡선 시각화 (Loss, Accuracy)
- [ ] MLflow 실험 기록 완료
- [ ] Validation Accuracy ~98% 달성
- [ ] 모델 저장 (.h5 + MLflow)

## 🎯 다음 단계 (Day 3-2)

**Day 3-2: 다양한 CNN 아키텍처 비교**

5가지 모델 구현 & 비교:
1. **LeNet-5**: Conv(6) → Pool → Conv(16) → Pool → FC
2. **VGG-style**: Conv×2 → Pool (3번 반복)
3. **ResNet-style**: Skip Connection으로 깊게
4. **SE-CNN**: Squeeze-Excitation Attention
5. **Custom**: Conv + Self-Attention Hybrid

**각 모델:**
- 완전한 구현
- MLflow 자동 로깅
- 성능 비교 테이블
- 학습 곡선 비교

**예상 결과:**
```
Simple CNN    : 98.0%
LeNet-5       : 98.2%
VGG-style     : 99.0%
ResNet-style  : 99.2%
Attention     : 99.3%
Custom        : 99.4%
```

축하합니다! Day 3-1 완료 🎉